# Day 6 Advanced Analytics

This notebook computes Historical VaR/CVaR, rolling 90-day Sharpe, investor cohort analytics, SIP continuity risk, a sector HHI concentration analysis, and advanced insights for the Bluestock MF project.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..')
PROC = ROOT / 'data' / 'processed'
OUT = ROOT / 'reports' / 'performance'
OUT.mkdir(parents=True, exist_ok=True)

nav = pd.read_csv(PROC / '02_nav_history_processed.csv', parse_dates=['date'])
perf = pd.read_csv(PROC / '07_scheme_performance_processed.csv')
tx = pd.read_csv(PROC / '08_investor_transactions_processed.csv', parse_dates=['transaction_date'])
holdings = pd.read_csv(PROC / '09_portfolio_holdings_processed.csv')

print('Loaded data:', nav.shape, perf.shape, tx.shape, holdings.shape)

Loaded data: (64320, 3) (40, 19) (32778, 13) (322, 8)


## Historical VaR and CVaR (95%)

Compute the 5th percentile of daily returns and the conditional mean of losses below that threshold for all schemes.

In [2]:
nav = nav.sort_values(['amfi_code', 'date']).copy()
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

var_rows = []
for code, group in nav.groupby('amfi_code'):
    returns = group['daily_return'].dropna()
    if returns.empty:
        continue
    var_95 = returns.quantile(0.05)
    cvar_95 = returns[returns <= var_95].mean() if not returns[returns <= var_95].empty else var_95
    var_rows.append({
        'amfi_code': code,
        'var_95': var_95,
        'cvar_95': cvar_95,
        'observations': int(len(returns)),
    })

var_cvar_df = pd.DataFrame(var_rows)
var_cvar_df = var_cvar_df.merge(perf[['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'risk_grade']], on='amfi_code', how='left')
var_cvar_df = var_cvar_df.sort_values(['var_95', 'amfi_code'])
var_cvar_df.to_csv(OUT / 'var_cvar_report.csv', index=False)
var_cvar_df.head(10)

,amfi_code,var_95,cvar_95,observations,scheme_name,fund_house,category,plan,risk_grade
4,101207,-0.023915,-0.030289,1607,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Small Cap,Regular,Very High
17,119095,-0.023284,-0.029690,1607,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Small Cap,Regular,Very High
22,119599,-0.023155,-0.030163,1607,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,Very High
11,118634,-0.022810,-0.029940,1607,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Small Cap,Regular,Very High
39,149324,-0.021520,-0.028573,1607,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,Small Cap,Regular,Very High
21,119598,-0.021502,-0.028444,1607,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,Very High
16,119094,-0.016997,-0.022375,1607,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,Mid Cap,Regular,High
29,120842,-0.016950,-0.021251,1607,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,Mid Cap,Regular,High
2,100033,-0.016902,-0.021850,1607,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Mid Cap,Regular,High
7,102886,-0.016857,-0.021771,1607,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,Mid Cap,Regular,High


## Rolling 90-day Sharpe Ratio

Calculate rolling Sharpe for the five largest schemes by AUM and export a time series chart.

In [3]:
top_funds = perf.sort_values('aum_crore', ascending=False).head(5)
selected = nav[nav['amfi_code'].isin(top_funds['amfi_code'])].copy()
selected = selected.sort_values(['amfi_code', 'date']).reset_index(drop=True)
selected['daily_return'] = selected.groupby('amfi_code')['nav'].pct_change()

def rolling_sharpe(series):
    mean = series.rolling(90, min_periods=60).mean()
    std = series.rolling(90, min_periods=60).std()
    return mean.div(std).mul(np.sqrt(252))

selected['rolling_sharpe'] = selected.groupby('amfi_code')['daily_return'].transform(rolling_sharpe)
pivot = selected.pivot(index='date', columns='amfi_code', values='rolling_sharpe')
plt.figure(figsize=(12, 6))
for code in pivot.columns:
    name = top_funds.loc[top_funds['amfi_code'] == code, 'scheme_name'].iloc[0]
    plt.plot(pivot.index, pivot[code], label=f"{name} ({code})")
plt.title('Rolling 90-day Sharpe Ratio for Top 5 AUM Funds')
plt.xlabel('Date')
plt.ylabel('Rolling Sharpe')
plt.legend(fontsize=8)
plt.grid(alpha=0.2)
plt.tight_layout()
chart_path = OUT / 'rolling_sharpe_chart.png'
plt.savefig(chart_path, dpi=200)
plt.close()
from IPython.display import Image, display
display(Image(str(chart_path)))
chart_path

PosixPath('../reports/performance/rolling_sharpe_chart.png')

## Investor Cohort Analysis

Group investors by their first transaction year and compute average SIP amount, total invested amount, and top fund preference for each cohort.

In [4]:
transaction_data = tx.copy()
transaction_data['cohort_year'] = transaction_data.groupby('investor_id')['transaction_date'].transform('min').dt.year

sip_transactions = transaction_data[transaction_data['transaction_type'] == 'SIP'].copy()
invested_transactions = transaction_data[transaction_data['transaction_type'].isin(['SIP', 'Lumpsum'])].copy()

cohort_summary = (
    sip_transactions.groupby('cohort_year')['amount_inr'].mean().rename('avg_sip_amount').to_frame()
    .join(invested_transactions.groupby('cohort_year')['amount_inr'].sum().rename('total_invested'))
    .reset_index()
)

top_fund_pref = (
    invested_transactions.groupby(['cohort_year', 'amfi_code'])['amount_inr'].sum().reset_index()
    .sort_values(['cohort_year', 'amount_inr'], ascending=[True, False])
    .groupby('cohort_year')
    .first()
    .reset_index()
)
top_fund_pref = top_fund_pref.merge(perf[['amfi_code', 'scheme_name']], on='amfi_code', how='left')
top_fund_pref = top_fund_pref.rename(columns={'scheme_name': 'top_fund_preference', 'amount_inr': 'top_fund_amount'})
cohort_summary = cohort_summary.merge(top_fund_pref[['cohort_year', 'top_fund_preference']], on='cohort_year', how='left')
cohort_summary

,cohort_year,avg_sip_amount,total_invested,top_fund_preference
0,2024,10996.885825,2258062304,Axis Small Cap Fund - Regular - Growth
1,2025,13505.209581,18992635,Axis Midcap Fund - Regular - Growth


## SIP Continuity Analysis

Analyze investors with at least 6 SIP transactions and flag those with average gaps above 35 days as at-risk.

In [5]:
sip_only = transaction_data[transaction_data['transaction_type'] == 'SIP'].copy()
sip_only = sip_only.sort_values(['investor_id', 'transaction_date']).reset_index(drop=True)
eligible = sip_only.groupby('investor_id').size() >= 6
eligible_investors = eligible[eligible].index
sips_eligible = sip_only[sip_only['investor_id'].isin(eligible_investors)].copy()
sips_eligible['gap_days'] = sips_eligible.groupby('investor_id')['transaction_date'].diff().dt.days
avg_gap = sips_eligible.groupby('investor_id')['gap_days'].mean().reset_index(name='avg_gap_days')
avg_gap['at_risk'] = avg_gap['avg_gap_days'] > 35
avg_gap.head()

,investor_id,avg_gap_days,at_risk
0,INV000004,85.400000,True
1,INV000008,70.400000,True
2,INV000010,64.800000,True
3,INV000011,40.166667,True
4,INV000012,57.000000,True


## Sector HHI Concentration

Compute Herfindahl-Hirschman Index for fund holdings by sector and compare concentration across equity funds.

In [6]:
holdings = holdings.copy()
holdings['weight_share'] = holdings['weight_pct'] / 100.0
hhi = holdings.groupby('amfi_code')['weight_share'].apply(lambda w: (w ** 2).sum()).reset_index(name='hhi')
hhi = hhi.merge(perf[['amfi_code', 'scheme_name', 'fund_house', 'category', 'risk_grade']], on='amfi_code', how='left')
hhi = hhi.sort_values('hhi', ascending=False)
hhi.head(10)

,amfi_code,hhi,scheme_name,fund_house,category,risk_grade
11,119092,0.206448,Axis Bluechip Fund - Regular - Growth,Axis Mutual Fund,Large Cap,Moderate
3,101207,0.200700,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Small Cap,Very High
18,119599,0.174751,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Very High
4,102885,0.174709,UTI Nifty 50 Index Fund - Regular - Growth,UTI Mutual Fund,Index,Moderate
7,118632,0.168298,Nippon India Large Cap Fund - Regular - Growth,Nippon India MF,Large Cap,Moderate
29,148568,0.167930,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,Large & Mid Cap,Moderately High
21,120505,0.157570,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,Mid Cap,High
22,120506,0.153794,ICICI Pru Value Discovery Fund - Regular - Growth,ICICI Prudential MF,Value,Moderately High
27,125498,0.152414,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,High
23,120841,0.149680,Kotak Bluechip Fund - Regular - Growth,Kotak Mahindra MF,Large Cap,Moderate


## Advanced Insights

1. The fund with the most negative 95% VaR is the most vulnerable under extreme daily loss scenarios.
2. Earlier investor cohorts contribute the highest total invested amounts, showing cohort persistence in the book.
3. SIP continuity is strongest for investors with average gaps under 35 days; flagged investors are at-risk for churn.
4. High HHI funds are the most sector-concentrated portfolios and should be monitored for concentration risk.
5. Top AUM funds also show resilient rolling Sharpe behavior, making them strong candidates for core allocation.